# 🧪 캐글 전용 통합 실험 조종석 (Kaggle Mini Experiment Notebook)

이 노트북은 **CLIP 장면 전환, OWL-ViT 객체 탐지 궤적(X, Y), 그리고 카메라 줌인/줌아웃(Area)** 피처를 전부 통합한 최종 힌트 모델(`v7_cot_hint`)을 캐글 GPU 환경에서 단 한 번에 구현하고 학습/평가하기 위한 통합 조종석입니다.

### 1. 깃허브 최신 저장소 복제 및 작업 경로 이동

In [ ]:
import os
if not os.path.exists("SNU_Challenge"):
    print("Cloning SNU_Challenge Repository...")
    !git clone https://github.com/Song-exp/SNU_Challenge.git

%cd SNU_Challenge

### 2. OWL-ViT 객체 탐지 및 시공간 궤적/면적 피처 전처리 추출 (약 5~10분 소요)
캐글의 T4 혹은 P100 GPU 가속을 활용해 전체 비디오 프레임에 존재하는 핵심 피사체의 궤적 정보를 자동으로 추출합니다.

In [ ]:
# GPU 가속 기반 OWL-ViT 피처 추출
!python scripts/extract_owlvit_features.py

### 3. 미니 CoT 힌트 LoRA 학습 구동 (약 1~1.5시간 소요)
Gemma가 텍스트 분석한 카메라 기법과 OWL-ViT가 이미지에서 찾아낸 면적/중심 좌표 변화를 연동한 `v7_cot_hint` 프롬프트로 1,000개 샘플 학습을 시작합니다.

In [ ]:
# Qwen3-VL-2B 로컬 경로 설정 (캐글에 모델을 올렸다면 해당 Input 경로로 변경 가능)
MODEL_PATH = "./models/Qwen3-VL-2B-Instruct"
if not os.path.exists(MODEL_PATH):
    # 캐글 데이터셋으로 붙였을 때의 통상 경로 예시 추가
    MODEL_PATH = "/kaggle/input/qwen3-vl-2b-instruct/Qwen3-VL-2B-Instruct" 

# 미니 LoRA 파인튜닝 실행
!python scripts/train_cot.py \
    --run-name mini_cot_hint_aug2 \
    --model {MODEL_PATH} \
    --prompt v7_cot_hint \
    --max-samples 1000 \
    --max-steps 300 \
    --events-from spacy

### 4. holdout 300 평가 데이터셋 검증 및 게이트 판정
학습이 끝난 어댑터를 붙여 baseline(15.5%) 대비 `+4%p` 이상 향상되는지 검증합니다.

In [ ]:
# 검증셋 평가 시작
!python scripts/eval_zero_shot.py \
    --model {MODEL_PATH} \
    --adapter ./outputs/runs/mini_cot_hint_aug2/adapter \
    --prompt v7_cot_hint